# Day 16: Demand Forecasting & What-If Analysis

This notebook contains demand forecasting visualizations (Prophet, LSTM, and Hybrid models) and implements a What-If Analysis framework to simulate different business scenarios.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# 1. Load mock forecast data
np.random.seed(42)
dates = pd.date_range(start="2026-07-01", periods=30)
trend = np.linspace(200, 300, 30)
seasonality = 50 * np.sin(np.linspace(0, 4*np.pi, 30))
noise = np.random.normal(0, 10, 30)

df_forecast = pd.DataFrame({
    "Date": dates,
    "Prophet": np.round(trend + seasonality, 2),
    "LSTM": np.round(trend + seasonality + noise * 1.5, 2),
    "Hybrid": np.round(trend + seasonality + noise * 0.5, 2)
})

print("Avg Daily Hybrid Forecast: $", round(df_forecast['Hybrid'].mean(), 2))
print("Peak Predicted Sales: $", round(df_forecast['Hybrid'].max(), 2))
df_forecast.head()

In [ ]:
# 2. Visualize Multi-Model Forecasts
df_melted = df_forecast.melt(id_vars=["Date"], value_vars=["Prophet", "LSTM", "Hybrid"], var_name="Model", value_name="Sales")
fig = px.line(
    df_melted,
    x="Date",
    y="Sales",
    color="Model",
    title="30-Day Out-of-Sample Sales Forecast",
    markers=True
)
fig.show()

## What-If Scenario Analysis

Here we simulate different market scenarios by shifting the forecast curves:
- **Optimistic Scenario**: 15% increase in demand (e.g., due to marketing promotion).
- **Pessimistic Scenario**: 20% decrease in demand (e.g., due to supply disruptions).

In [ ]:
# Define multipliers
optimistic_multiplier = 1.15
pessimistic_multiplier = 0.80

df_scenarios = df_forecast[["Date", "Hybrid"]].copy()
df_scenarios["Base_Forecast"] = df_scenarios["Hybrid"]
df_scenarios["Optimistic_Scenario"] = np.round(df_scenarios["Hybrid"] * optimistic_multiplier, 2)
df_scenarios["Pessimistic_Scenario"] = np.round(df_scenarios["Hybrid"] * pessimistic_multiplier, 2)

print("Total Projected Sales - Base: $", round(df_scenarios["Base_Forecast"].sum(), 2))
print("Total Projected Sales - Optimistic: $", round(df_scenarios["Optimistic_Scenario"].sum(), 2))
print("Total Projected Sales - Pessimistic: $", round(df_scenarios["Pessimistic_Scenario"].sum(), 2))
df_scenarios.head()

In [ ]:
# Plot Scenarios
fig_scenarios = go.Figure()
fig_scenarios.add_trace(go.Scatter(x=df_scenarios["Date"], y=df_scenarios["Base_Forecast"], name="Base (Hybrid)", line=dict(color="blue", width=2)))
fig_scenarios.add_trace(go.Scatter(x=df_scenarios["Date"], y=df_scenarios["Optimistic_Scenario"], name="Optimistic (+15%)", line=dict(color="green", width=2, dash="dash")))
fig_scenarios.add_trace(go.Scatter(x=df_scenarios["Date"], y=df_scenarios["Pessimistic_Scenario"], name="Pessimistic (-20%)", line=dict(color="red", width=2, dash="dot")))

fig_scenarios.update_layout(
    title="What-If Demand Scenarios Comparison",
    xaxis_title="Date",
    yaxis_title="Predicted Sales ($)",
    hovermode="x unified"
)
fig_scenarios.show()